In [1]:
# Importa tudo

from selenium import webdriver
from selenium.webdriver.support.select import Select
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.common.by import By
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.alert import Alert
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import NoSuchElementException
from selenium.common.exceptions import TimeoutException, WebDriverException
from webdriver_manager.chrome import ChromeDriverManager

#Bibliotecas de Sistema
import time
import re
import csv
import os
import psutil
from bs4 import BeautifulSoup
from eproc_driver import eproc as eproc
import sqlite3
from pathlib import Path
import io
import pandas as pd
from contextlib import closing

#bibliotecas de configuração
import pyotp
import configparser
import keyring

#bibliotecas de automação
import pyperclip
import pyautogui

#Bibliotecas de IA
from gemini import gemini as gemini
import ollama

In [2]:
# Importa o essencial

#Bibliotecas de Sistema
import time
import re
import csv
import os
import psutil
from bs4 import BeautifulSoup
from eproc_driver import eproc as eproc
import sqlite3
from pathlib import Path
import io
import pandas as pd
from contextlib import closing

#bibliotecas de configuração
import pyotp
import configparser
import keyring

#bibliotecas de automação
import pyperclip
import pyautogui

#Bibliotecas de IA
from gemini import gemini as gemini
import ollama

#pasta_downloads = r"C:\Users\dodonin\Downloads"
pasta_downloads = r"D:\Downloads"

navegador = eproc.novo_browser(pasta_downloads)

#configura variáveis
username = "dodonin"
password = keyring.get_password("eproc", username)
pyotop_code = "GJRGIYTCGBSGKYTEHE2TOZRUGFQTMMRQ"

#eproc.login_no_eproc_tj(navegador, username, password, pyotop_code)
eproc.login_no_eproc(navegador, username, password, pyotop_code)


Driver do Eproc importado


In [3]:
# Entra no perfil da Vara
perfil = "SRD1CIV"


eproc.entrar_no_perfil(navegador, perfil)

Perfil carregado: SRD1CIV


In [23]:
# Importa o arquivo "Processos.csv" para uma tabela chamada {perfil}_control

df_processos = pd.read_csv("Processos.csv", dtype=str)
df_processos["ok"] = 0

with sqlite3.connect("movimentos.db") as conn:
    df_processos.rename(columns={df_processos.columns[0]: "num_processo"}, inplace=True)
    df_processos[["num_processo", "ok"]].to_sql(f"{perfil}_control", conn, if_exists="replace", index=False)

In [4]:
with sqlite3.connect("movimentos.db") as conn:
    df_pendentes = pd.read_sql_query(f"SELECT * FROM {perfil}_control WHERE ok = 0", conn)
df_pendentes = df_pendentes["num_processo"].tolist()
print(df_pendentes)

['5000005-84.2008.8.21.0069', '5000007-97.2021.8.21.0069', '5000025-46.2006.8.21.0069', '5000026-02.2004.8.21.0069', '5000028-73.2021.8.21.0069', '5000046-07.2015.8.21.0069', '5000072-49.2008.8.21.0069', '5000077-37.2009.8.21.0069', '5000088-95.2011.8.21.0069', '5000090-36.2009.8.21.0069', '5000091-06.2018.8.21.0069', '5000093-98.2003.8.21.0069', '5000098-08.2012.8.21.0069', '5000099-27.2011.8.21.0069', '5000099-61.2010.8.21.0069', '5000100-12.2011.8.21.0069', '5000101-65.2009.8.21.0069', '5000101-89.2014.8.21.0069', '5000111-22.2003.8.21.0069', '5000112-07.2003.8.21.0069', '5000117-19.2009.8.21.0069', '5000119-23.2008.8.21.0069', '5000120-32.2013.8.21.0069', '5000133-60.2015.8.21.0069', '5000154-75.2011.8.21.0069', '5000171-14.2011.8.21.0069', '5000184-13.2011.8.21.0069', '5000189-06.2009.8.21.0069', '5000201-10.2015.8.21.0069', '5000205-23.2010.8.21.0069', '5000216-66.2021.8.21.0069', '5000229-75.2015.8.21.0069', '5000230-65.2012.8.21.0069', '5000247-62.2016.8.21.0069', '5000257-77.2

In [ ]:
# Rotina principal

# Lista de data-nome indesejáveis
tipos_indesejaveis = ["PROC"]

for processo in df_pendentes:
    try:
        eproc.entrar_no_processo(navegador, processo)
    except:
        print(f"erro ao entrar no processo {processo}")
    
    try:
        eproc.pega_eventos(navegador, perfil, processo)
    except:
        print(f"erro ao pegar eventos do processo {processo}")
        
    try:
        eproc.atualiza_textos_documentos(navegador, processo, tipos_indesejaveis)
    except:
        print(f"erro ao atualizar documento do processo {processo}")
    
    conn = sqlite3.connect("movimentos.db")
    cursor = conn.cursor()
    
    print(f"Processo {processo} atualizado.")

    # Verifica se ao menos um evento do processo possui o campo 'documentos' preenchido
    cursor.execute(
        f"SELECT 1 FROM movimentos WHERE processo = ? AND documentos IS NOT NULL AND documentos != '' LIMIT 1",
        (processo,)
    )
    if cursor.fetchone():
        cursor.execute(f"""
            UPDATE {perfil}_control SET ok = 1 WHERE num_processo = ?
        """, (processo,))

    conn.commit()
    cursor.close()
    conn.close()

Página do processo 5000005-84.2008.8.21.0069 carregada com sucesso.
0 movimentos do processo 5000005-84.2008.8.21.0069 inseridos no banco de dados.
Processo 5000005-84.2008.8.21.0069 atualizado.
Página do processo 5000007-97.2021.8.21.0069 carregada com sucesso.
0 movimentos do processo 5000007-97.2021.8.21.0069 inseridos no banco de dados.
Processo 5000007-97.2021.8.21.0069 atualizado.
Página do processo 5000025-46.2006.8.21.0069 carregada com sucesso.
0 movimentos do processo 5000025-46.2006.8.21.0069 inseridos no banco de dados.
Processo 5000025-46.2006.8.21.0069 atualizado.
Página do processo 5000026-02.2004.8.21.0069 carregada com sucesso.
0 movimentos do processo 5000026-02.2004.8.21.0069 inseridos no banco de dados.
Documentos do evento 78 do processo 5000026-02.2004.8.21.0069 atualizados.
Documentos do evento 75 do processo 5000026-02.2004.8.21.0069 atualizados.
Documentos do evento 74 do processo 5000026-02.2004.8.21.0069 atualizados.
Documentos do evento 74 do processo 500002

In [7]:
# RESETA OS STATUS

conn = sqlite3.connect("movimentos.db")
cursor = conn.cursor()

cursor.execute("UPDATE control SET ok = 0")
conn.commit()

cursor.close()
conn.close()

In [6]:
# DELETA TUDO

conn = sqlite3.connect("movimentos.db")
cursor = conn.cursor()

cursor.execute("DELETE FROM movimentos")
conn.commit()

cursor.close()
conn.close()

In [ ]:
# TESTE

processo = "5004484-95.2023.8.21.0069"
# Lista de data-nome indesejáveis
tipos_indesejaveis = ["PROC"]

eproc.entrar_no_processo(navegador, processo)
eproc.pega_eventos(navegador, perfil, processo)
eproc.atualiza_textos_documentos(navegador, processo, tipos_indesejaveis)

conn = sqlite3.connect("movimentos.db")
cursor = conn.cursor()

print(f"Processo {processo} atualizado.")
cursor.execute("""
    UPDATE control SET ok = 1 WHERE num_processo = ?
""", (processo,))

conn.commit()
cursor.close()
conn.close()

Página do processo 5004484-95.2023.8.21.0069 carregada com sucesso.
0 movimentos do processo 5004484-95.2023.8.21.0069 inseridos no banco de dados.
Documentos do evento 23 do processo 5004484-95.2023.8.21.0069 atualizados.
Documentos do evento 20 do processo 5004484-95.2023.8.21.0069 atualizados.
Documentos do evento 17 do processo 5004484-95.2023.8.21.0069 atualizados.
Documentos do evento 17 do processo 5004484-95.2023.8.21.0069 atualizados.
Documentos do evento 17 do processo 5004484-95.2023.8.21.0069 atualizados.
Documentos do evento 17 do processo 5004484-95.2023.8.21.0069 atualizados.
Documentos do evento 17 do processo 5004484-95.2023.8.21.0069 atualizados.
Documentos do evento 17 do processo 5004484-95.2023.8.21.0069 atualizados.
Documentos do evento 17 do processo 5004484-95.2023.8.21.0069 atualizados.
Documentos do evento 17 do processo 5004484-95.2023.8.21.0069 atualizados.
Documentos do evento 17 do processo 5004484-95.2023.8.21.0069 atualizados.
Documentos do evento 17 do 

In [ ]:
# Faz um resumo do conteúdo usando o Gemma + as minutas da Vara
conteudo = "oi"
pergunta_gemma = "Responda objetivamente:" \
"1. quem é o autor do documento?" \
"2. qual é o pedido feito (se for um pedido)? qual foi a decisão (se foi uma decisão)?" \
"3. qual é o próximo passo sugerido, considerando a lei brasileira?" \

resumo = ollama.chat(
    model="cnmoro/gemma3-gaia-ptbr-4b:q8_0",
    messages=[
        {"role": "system", "content": pergunta_gemma},
        {"role": "user", "content": conteudo}
    ]
)['message']['content']

print(resumo)

Para responder à sua pergunta de forma eficaz, preciso do documento em questão. Como você não me forneceu nenhum documento, não posso identificar o autor, o pedido, a decisão ou o próximo passo sugerido. 

Por favor, compartilhe o documento com o qual você está trabalhando. Uma vez que eu o tiver, eu farei o meu melhor para analisar e responder às suas perguntas objetivamente.



In [5]:
conn = sqlite3.connect("movimentos.db")
cursor = conn.cursor()

In [16]:
eventos = [el for el in navegador.find_elements(By.CLASS_NAME, "infraEventoDescricao") if el.tag_name == "label"]

for evento in eventos:
    texto_evento = evento.text.split('\n')[0]
    tr_element = evento.find_element(By.XPATH, "./ancestor::tr")
    # Pega o texto da segunda coluna (td[2]) de tr_element
    evento_text = tr_element.find_element(By.XPATH, './td[2]').text
    conn = sqlite3.connect("movimentos.db")
    cursor = conn.cursor()
    cursor.execute(
        "SELECT 1 FROM movimentos WHERE processo = ? AND evento = ? LIMIT 1",
        (str(processo), int(evento_text))
    )
    existe = cursor.fetchone() is not None
    if not existe:
        cursor.execute(
            "INSERT INTO movimentos (processo, evento, vara, descricao) VALUES (?, ?, ?, ?)",
            (str(processo), int(evento_text), perfil, texto_evento)
        )
        conn.commit()


In [35]:
conn = sqlite3.connect("movimentos.db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS movimentos (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    processo TEXT,
    vara TEXT,
    evento INTEGER,
    descricao TEXT,
    usuario TEXT,
    documentos TEXT,
    resumo TEXT,
    encaminhamento TEXT
)
""")

conn.commit()
conn.close()

In [34]:
conn = sqlite3.connect("movimentos.db")
cursor = conn.cursor()

cursor.execute("""
DROP TABLE movimentos
""")

conn.commit()
conn.close()

In [7]:
conn.close()